In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

In [2]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

Using device: cuda


In [3]:
def compute_growth(stage_score, ripeness_score=None):
    """
    Returns growth percentage (0–100)
    """

    # Case A: no fruit detected (early stage)
    if ripeness_score is None:
        return stage_score * 60.0

    # Case B: fruit detected
    return 0.4 * stage_score * 100 + 0.6 * ripeness_score * 100


In [4]:
# [stage_score, ripeness_score or -1]
X = np.array([
    [0.2, -1],   # early veg, no fruit
    [0.4, -1],
    [0.6, -1],
    [0.6, 0.3],  # early fruit
    [0.7, 0.5],
    [0.8, 0.7],
    [0.9, 0.9],
    [1.0, 1.0],  # fully mature
], dtype=np.float32)

# Ground-truth growth %
y = np.array([
    12, 24, 36, 55, 68, 82, 92, 100
], dtype=np.float32)

# Replace -1 ripeness with 0 for training
X[X[:,1] < 0, 1] = 0.0

X = torch.tensor(X).to(DEVICE)
y = torch.tensor(y).unsqueeze(1).to(DEVICE)


In [5]:
class GrowthRegressor(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 1)
        )

    def forward(self, x):
        return self.net(x)


In [6]:
model = GrowthRegressor().to(DEVICE)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [7]:
EPOCHS = 1000

for epoch in range(EPOCHS):
    model.train()
    preds = model(X)
    loss = criterion(preds, y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 100 == 0:
        print(f"Epoch {epoch} | Loss: {loss.item():.4f}")


Epoch 0 | Loss: 4380.0151
Epoch 100 | Loss: 4358.7993
Epoch 200 | Loss: 4278.8730
Epoch 300 | Loss: 3981.9407
Epoch 400 | Loss: 3303.4270
Epoch 500 | Loss: 2295.8284
Epoch 600 | Loss: 1261.3744
Epoch 700 | Loss: 541.6516
Epoch 800 | Loss: 232.0141
Epoch 900 | Loss: 152.2951


In [8]:
import os

SAVE_PATH = "models/growth_regressor.pth"
os.makedirs("models", exist_ok=True)

torch.save(model.state_dict(), SAVE_PATH)
print("Growth model saved at:", SAVE_PATH)


Growth model saved at: models/growth_regressor.pth
